# PoliMillionaire Game Tests

Run live games with selected local models, one after another.

No generated-answer API is used.


## Setup

Find the repo and local model cache.


In [1]:
import gc
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire
HF cache: /Users/camiloa2m/Downloads/NLP/nlp-polimillionaire/data/hf_home


## Settings

Pick models here. Each selected model gets its own game run.


In [2]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_ID = 0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen 2.5 3B", "model_id": "Qwen/Qwen2.5-3B-Instruct", "run": True},
]

SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

print("API check:", RUN_API_CHECK)
print("Live game:", RUN_LIVE_GAME)
print("Competition:", COMPETITION_ID)
print("Models:")
for item in MODELS_TO_RUN:
    print("-", item["label"], item["model_id"], "run=", item["run"])


API check: True
Live game: True
Competition: 0
Models:
- Gemma 4 E2B google/gemma-4-E2B-it run= False
- Gemma 4 E4B google/gemma-4-E4B-it run= False
- Qwen 2.5 3B Qwen/Qwen2.5-3B-Instruct run= True


## Login

Use environment variables, Colab secrets, or prompt mode.


In [3]:
import getpass
from dotenv import load_dotenv
from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
load_dotenv()

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")

try:
    from google.colab import userdata

    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass

if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")

print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))


Username configured: True
Password configured: True


## Competitions

Turn on `RUN_API_CHECK` to list games.


In [4]:
client = None

if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")


Logged in as promptPotamo
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


## Run Selected Models

Each model warms up, plays if enabled, then unloads.


In [5]:
!pip install -qU langchain-ollama
!pip install  -qU colab-xterm
%load_ext colabxterm
# It may ask you to restart, accept and run this cell again

### For Colab: Run the following commands on the terminal below:
To spawn the server deamon please run the cell with ```%xterm```. It will create a concurrent shell where we can paste the following commands but, at the same time, also run other cells!

```sudo apt-get install zstd```

```curl -fsSL https://ollama.com/install.sh | sh```

```ollama serve & ollama pull llama3```

In [6]:
# %xterm

In [7]:
# from langchain_ollama import ChatOllama
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.tools import tool
# from langchain_core.tools import render_text_description
# from langchain_core.output_parsers import JsonOutputParser
# from typing import Any, Dict, Optional, TypedDict
# from langchain_core.runnables import RunnableConfig
# from langchain_core.runnables import RunnablePassthrough

# @tool
# def compound_interest(principal: float, rate: float, years: float) -> float:
#    """
#    Computes result of applying particular interest rate to a principal for a number of years.
#    First argument is the principal, second is the rate, and the third is the number of years.
#    Equation calculated is: output = principal * (1+rate/100)^years.
#    """
#    return principal * ((1+rate/100) ** years)

# @tool
# def exponentiate(base: float, power: float) -> float:
#    """Computes base^power, i.e. raises the first argument to the power of the second argument."""
#    return base ** power

# print("Testing tools:")
# print(compound_interest.invoke({"principal": 50, "rate": 95, "years": 10}))
# print(exponentiate.invoke({"base": 1.95, "power": 10}))
# print("--------------------------------")

# tools = [compound_interest, exponentiate]
# # Render the tools information in a human-friendly way
# print("Available tools:")
# rendered_tools = render_text_description(tools) 
# print(rendered_tools)
# print("--------------------------------")

# llm = ChatOllama(
#     model="qwen2.5:7b-instruct-q4_K_M",
#     temperature=0,
# )

# #system_prompt = """You are a helpful assistant. Answer all questions to the best of your ability."""
# system_prompt = f"""\
# You are a helpful assistant. Answer all questions to the best of your ability.
# You has access to the following set of tools.
# Here are the names and descriptions for each tool:

# {rendered_tools}

# If a question is asking for recent infomation that you don't have access to, or if a question is asking for a calculation, you should use the tools above to get the answer.

# Given the user input, return the name and input of the tool to use.
# Return your response as a JSON blob with 'name' and 'arguments' keys.

# The `arguments` should be a dictionary, with keys corresponding
# to the argument names and the values corresponding to the requested values.
# """

# prompt = ChatPromptTemplate.from_messages([
#     ("system", system_prompt),
#     ("user", "{input}")
# ])


# chain = prompt | llm

# # message = {"input": "What is the capital of France?"}
# message = {"input": "At an annual interest rate of 95% interest, what would 50 dollars be worth in 10 years time?"}

# response = chain.invoke(message)

# print("Response structure:\n", dict(response))
# print("--------------------------------")
# print("Response:", response.content)

# print("--------------------------------")
# chain_with_parser = prompt | llm | JsonOutputParser()
# response = chain_with_parser.invoke(message)
# print("Parsed response structure:\n", response)
# print("--------------------------------")

# # This is an example of how we can take the output from the LLM and use it to invoke a tool.
# class ToolCallRequest(TypedDict):
#     """A typed dict that shows the inputs into the invoke_tool function."""
#     name: str
#     arguments: Dict[str, Any]

# def invoke_tool(tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None):
#     """A function that we can use the perform a tool invocation.

#     Args:
#         tool_call_request: a dict that contains the keys name and arguments.
#             The name must match the name of a tool that exists.
#             The arguments are the arguments to that tool.
#         config: This is configuration information that LangChain uses that contains
#             things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

#     Returns:
#         output from the requested tool
#     """
#     tool_name_to_tool = {tool.name: tool for tool in tools}
#     name = tool_call_request["name"]
#     requested_tool = tool_name_to_tool[name]
#     return requested_tool.invoke(tool_call_request["arguments"], config=config)

# # This is how we can create a chain that goes all the way from prompt to LLM to output parser to tool invocation.
# chain_with_parser_and_invoker = prompt | llm | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)

# print("---------------------------------")
# print("Invoking chain with tool invocation...")
# response = chain_with_parser_and_invoker.invoke(message)
# response



In [8]:
from dataclasses import dataclass, field
from typing import Any

"""Small dataclasses shared by the assignment notebook helpers."""

@dataclass(frozen=True)
class AnswerOption:
    id: int
    text: str


@dataclass(frozen=True)
class Question:
    id: int
    text: str
    options: list[AnswerOption]
    level: int = 0
    metadata: dict[str, Any] = field(default_factory=dict)

    def valid_option_ids(self) -> set[int]:
        return {option.id for option in self.options}

    def first_option(self) -> AnswerOption:
        if not self.options:
            raise ValueError("Question has no answer options")
        return self.options[0]

    def get_option(self, option_id: int) -> AnswerOption | None:
        for option in self.options:
            if option.id == option_id:
                return option
        return None

    def require_option(self, option_id: int) -> AnswerOption:
        return self.get_option(option_id) or self.first_option()


@dataclass
class AnswerPrediction:
    option_id: int
    answer_text: str
    confidence: float | None = None
    reasoning: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)



In [ ]:
#* ============================================================
#* INSTALL (run once if needed)

# !pip install -q \
#     langchain \
#     langchain-core \
#     langchain-community \
#     langchain-ollama \
#     wikipedia \
#     ddgs\
#     numexpr \
#     pydantic \
#     rank_bm25



#* ============================================================
#* IMPORTS

import re
import wikipedia
import numexpr as ne

from enum import Enum
from typing import List

from ddgs import DDGS
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage,
    SystemMessage
)
from langchain_core.prompts import ChatPromptTemplate


#* ============================================================
#* CONFIG

MODEL_NAME = "qwen2.5:7b-instruct-q4_K_M"
TEMPERATURE = 0
MAX_DDG_RESULTS = 3


#* ============================================================
#* OUTPUT SCHEMA

class TriviaAnswer(BaseModel):
    answer: int = Field(
        description="Final answer, MCQ option integer ID"
    )
    confidence: int = Field(
        ge=0,
        le=100,
        description="Confidence score between 0 and 100"
    )
    evidence: str = Field(
        description="Short evidence-based justification"
    )


#* ============================================================
#* ROUTING

class Route(str, Enum):
    MATH = "math"
    GENERAL = "general"


#* ============================================================
#* LLM

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
)

structured_llm = llm.with_structured_output(
    TriviaAnswer
)


#* ============================================================
#* TOOLS

@tool
def duckduckgo_search(query: str) -> str:
    """
    Search the web for recent or time-sensitive information.
    """
    try:
        snippets = []
        with DDGS() as ddgs:
            results = ddgs.text(
                query,
                max_results=MAX_DDG_RESULTS
            )
            for r in results:
                snippets.append(
                    f"TITLE: {r.get('title', '')}\n"
                    f"SNIPPET: {r.get('body', '')}"
                )
        return "\n\n".join(snippets)

    except Exception as e:
        return f"DuckDuckGo error: {str(e)}"

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.

    Examples:
    - 2 + 2
    - 50 * (1 + 0.95)**10
    - sqrt(144)
    """

    try:
        result = ne.evaluate(expression)
        return str(result.item())

    except Exception as e:
        return f"Calculator error: {str(e)}"

#* ============================================================
#* TOOL REGISTRY

TOOLS = [
    duckduckgo_search,
    calculator
]

TOOL_MAP = {
    tool.name: tool
    for tool in TOOLS
}


#* ============================================================
#* ROUTER

MATH_PATTERNS = [
    r"\d+\s*[\+\-\*/]\s*\d+",
    r"interest",
    r"percentage",
    r"percent",
    r"probability",
    r"average",
    r"mean",
    r"median",
    r"solve",
    r"equation",
    r"sqrt",
    r"log",
    r"area",
    r"volume",
    r"calculate"
]

def matches_patterns(
    text: str,
    patterns: List[str]
) -> bool:

    text = text.lower()

    return any(
        re.search(pattern, text)
        for pattern in patterns
    )


def classify_question(question: str) -> Route:
    """
    Classify the question into a route based on keyword patterns.
    """
    if matches_patterns(
        question,
        MATH_PATTERNS
    ):
        return Route.MATH

    return Route.GENERAL


#* ============================================================
#* RETRIEVAL

def retrieve_evidence(
    question: str,
    route: Route
) -> str:
    
    print(f"Original Query: {question}")

    if route == Route.MATH:
        print("\n[PIPELINE] -> CALCULATOR")
        return ""


    print("\n[PIPELINE] -> DUCKDUCKGO")

    return duckduckgo_search.invoke({
        "query": question
    })



#* ============================================================
#* SYSTEM PROMPT


SYSTEM_PROMPT = """
You are a high-accuracy retrieval-augmented trivia assistant.

Rules:
1. Use retrieved evidence whenever available.
2. Never hallucinate recent information.
3. If multiple-choice options exist, select exactly one option.
4. Keep reasoning concise and factual.
5. If evidence is weak, lower confidence appropriately.
6. If no evidence is provided, prefer "unknown" and low confidence.
"""
#5. Base answers on evidence, not memorization.

#* SYSTEM PROMPT FOR MATH TOOL
#* ============================================================

SYSTEM_PROMPT_MATH_TOOL = """Extract ONLY the mathematical expression needed to solve the question.

Return ONLY the expression.

Examples:
2 + 2
50 * (1 + 0.95)**10
sqrt(144)
"""


#* REASONING PROMPT
#* ============================================================

REASONING_PROMPT = """QUESTION:
{question}

RETRIEVED EVIDENCE:
{evidence}

Answer the question using the evidence."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", REASONING_PROMPT)
])

reasoning_chain = prompt | structured_llm


def build_MCQ_string(question: Question) -> str:
    options = "\n".join(f"{option.id}. {option.text}" for option in question.options)
    return "\n\n".join([f"{question.text}" ,options])

#* ============================================================
#* MAIN PIPELINE

def ask_question(question: Question) -> TriviaAnswer:
    print("=" * 80)
    print("QUESTION")
    print("=" * 80)

    mcquestion_str = build_MCQ_string(question)
    print(mcquestion_str)

    #* ROUTING

    route = classify_question(question.text)
    print(f"\n[ROUTER] -> {route.value.upper()}")
 
    #* MATH PIPELINE

    calculator_result = None
    if route == Route.MATH:
        messages = [
            SystemMessage(content=SYSTEM_PROMPT_MATH_TOOL),
            HumanMessage(content=question.text)
        ]

        expression_response = llm.invoke(messages)
        expression = expression_response.content.strip()

        print("\n[EXTRACTED EXPRESSION]")
        print(expression)

        calculator_result = calculator.invoke({
            "expression": expression
        })

        evidence = (
            f"Calculated expression:\n"
            f"{expression}\n\n"
            f"Result:\n{calculator_result}"
        )

    #* FACTUAL / RECENT PIPELINE
    else:
        evidence = retrieve_evidence(
            question.text,
            route
        )

    #* EVIDENCE DISPLAY

    print("\n" + "=" * 80)
    print("RETRIEVED EVIDENCE")
    print("=" * 80)

    print("Full evidence length:", len(evidence), "characters")
    print(evidence[:2000])

    #* FINAL STRUCTURED REASONING

    result = reasoning_chain.invoke({
        "question": mcquestion_str,
        "evidence": evidence
    })

    #* FINAL OUTPUT

    print("\n" + "=" * 80)
    print("FINAL STRUCTURED OUTPUT")
    print("=" * 80)

    print(result.model_dump_json(indent=2))

    return result



# EXAMPLES

# WARMUP QUESTION
question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


# # FACTUAL QUESTION
# question = Question(
#     id=2,
#     text="Which country won the FIFA World Cup in 2018?",
#     options=[
#         AnswerOption(0, "Germany"),
#         AnswerOption(1, "Argentina"),
#         AnswerOption(2, "France"),
#         AnswerOption(3, "Brazil"),
#     ],
# )


# # RECENT QUESTION
# question = Question(
#     id=3,
#     text="Who won the UEFA Champions League in 2025?",
#     options=[
#         AnswerOption(0, "Real Madrid"),
#         AnswerOption(1, "Manchester City"),
#         AnswerOption(2, "PSG"),
#         AnswerOption(3, "Bayern Munich"),
#     ],
# )


# # MATH QUESTION
# question = """
# At an annual interest rate of 95%,
# what would 50 dollars be worth in 10 years?
# """


result = ask_question(question)


QUESTION
What is 2 + 2?

0. 3
1. 4
2. 5
3. 22

[ROUTER] -> MATH

[EXTRACTED EXPRESSION]
2 + 2

RETRIEVED EVIDENCE
Full evidence length: 39 characters
Calculated expression:
2 + 2

Result:
4

FINAL STRUCTURED OUTPUT
{
  "answer": 1,
  "confidence": 100.0,
  "evidence": "Calculated expression: 2 + 2. Result: 4."
}


In [ ]:
from datetime import datetime, timezone

from polimillionaire.runner import GameRunner, RunLogger
from polimillionaire.strategies import GemmaLLMConfig, GemmaStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
)


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception as exc:
        print("Cleanup warning:", exc)


def make_strategy(model_id: str) -> GemmaStrategy:
    config = GemmaLLMConfig(
        model_id=model_id,
        inference_backend="auto_model",
        device_map="auto",
        dtype="auto",
        max_new_tokens=8,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        seed=42,
        generation_max_time_seconds=18.0,
        timeout_seconds=120.0,
    )
    return GemmaStrategy(model_config=config)


def clean_name(label: str) -> str:
    return "_".join(label.lower().split())


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue

    strategy = None
    try:
        print()
        print("Model:", item["label"])
        strategy = make_strategy(item["model_id"])

        warmup = strategy.answer(warmup_question)
        print("warmup option:", warmup.option_id, warmup.answer_text)
        print("fallback:", warmup.metadata.get("fallback"))
        print("device:", warmup.metadata.get("device"))
        if warmup.metadata.get("fallback"):
            raise RuntimeError(f"Warm-up failed for {item['label']}.")

        result = {"label": item["label"], "model_id": item["model_id"], "warmup_option": warmup.option_id}

        if RUN_LIVE_GAME:
            run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            log_path = REPO_ROOT / "results" / "runs" / f"{clean_name(item['label'])}_{run_id}.jsonl"
            runner = GameRunner(
                client=client,
                safe_delay_seconds=SAFE_DELAY_SECONDS,
                answer_timeout_seconds=ANSWER_TIMEOUT_SECONDS,
                logger=RunLogger(log_path),
            )
            game = runner.play(COMPETITION_ID, strategy)
            result.update(
                {
                    "current_level": game.current_level,
                    "earned_amount": game.earned_amount,
                    "log_path": str(log_path),
                }
            )
            print("Reached level:", game.current_level)
            print("Earned amount:", game.earned_amount)
            print("Log path:", log_path)
        else:
            print("Live game skipped for", item["label"])

        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
        print("Cleared model memory after", item["label"])

run_results


## Results

Read recent game logs.


In [ ]:
from polimillionaire.runner import load_jsonl, summarize_attempts

run_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)

for path in run_logs[-5:]:
    print(path.name)

if run_logs:
    rows = load_jsonl(run_logs[-1])
    print("latest:", run_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No logs found.")


## Done

Use `run_results` and the JSONL logs for comparison.
